# Scientific TLDR Generator
### Fine-tuned BART + Diverse Beam Search + AlignScore

This notebook demonstrates the complete live pipeline:

1. Enter a scientific abstract.
2. Generate a baseline summary using greedy decoding.
3. Generate five candidates using Diverse Beam Search (DBS).
4. Score each candidate against the original abstract using AlignScore.
5. Select and compare the highest-scoring candidate.

**Selected validation configuration:** diversity penalty = **1.0**.

The live example is separate from the aggregate test-set results shown at the end.

## 1. Install dependencies

Run this in a fresh environment if packages are missing. A GPU is recommended.

AlignScore installation and checkpoint instructions can vary by version. Use the same AlignScore implementation and checkpoint as your experiment.

**Kernel:** choose `.venv-scoring` (transformers below 4.57). With transformers 4.57+ diverse beam search is not available and the DBS cell stops with instructions.

In [ ]:
# Do NOT run "pip install -U transformers ...": it can pull in transformers 4.57+ or 5.x, which breaks
# diverse beam search. Run this notebook with the .venv-scoring kernel (see SETUP_SCORING.md); it already has
# everything this notebook imports, including AlignScore.

In [ ]:
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Configure your models

Replace the Hugging Face repository with the exact location of your fine-tuned BART checkpoint. Optionally pin the repository to a commit hash so every group member loads the same model version.

In [ ]:
# REQUIRED: your uploaded fine-tuned BART model and tokenizer.
HF_MODEL_ID = "YOUR_HUGGINGFACE_USERNAME/YOUR_MODEL_REPOSITORY"
HF_REVISION = None  # Optional: exact Hugging Face commit hash

# Match the settings used in your experiment.
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 64
NUM_BEAMS = 10
NUM_BEAM_GROUPS = 5
NUM_RETURN_SEQUENCES = 5
DIVERSITY_PENALTY = 1.0

# REQUIRED: local AlignScore checkpoint path.
ALIGNSCORE_CHECKPOINT = "./AlignScore-large.ckpt"
ALIGNSCORE_MODEL = "roberta-large"
ALIGNSCORE_EVALUATION_MODE = "nli_sp"

## 3. Load fine-tuned BART

The same checkpoint is used for greedy decoding and DBS. The decoding strategy is the main difference between the two methods.

In [ ]:
if HF_MODEL_ID.startswith("YOUR_"):
    raise ValueError("Set HF_MODEL_ID to your actual Hugging Face repository.")

load_kwargs = {"revision": HF_REVISION} if HF_REVISION else {}
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, **load_kwargs)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(HF_MODEL_ID, **load_kwargs)
bart_model.to(DEVICE)
bart_model.eval()

print("Fine-tuned BART loaded.")

## 4. Enter a scientific abstract

Paste an abstract below. For a presentation, choose an abstract similar to the examples in SciTLDR-A.

In [ ]:
import ipywidgets as widgets

abstract_input = widgets.Textarea(
    placeholder="Paste a scientific abstract here...",
    description="Abstract:",
    layout=widgets.Layout(width="100%", height="220px")
)
display(abstract_input)

## 5. Generate the greedy baseline

Greedy decoding selects the highest-probability next token at each generation step and produces one summary.

In [ ]:
def tokenize_abstract(abstract):
    return tokenizer(
        abstract,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH
    ).to(DEVICE)


@torch.inference_mode()
def generate_greedy(abstract):
    inputs = tokenize_abstract(abstract)
    output_ids = bart_model.generate(
        **inputs,
        num_beams=1,
        num_return_sequences=1,
        do_sample=False,
        max_length=MAX_TARGET_LENGTH
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


def show_greedy(_):
    abstract = abstract_input.value.strip()
    if not abstract:
        print("Enter an abstract first.")
        return
    display(Markdown("### Greedy BART Summary"))
    display(Markdown(generate_greedy(abstract)))


greedy_button = widgets.Button(description="Generate Greedy Summary", button_style="primary")
greedy_button.on_click(show_greedy)
display(greedy_button)

## 6. Generate Diverse Beam Search candidates

DBS uses multiple beam groups to encourage different candidate sequences. This demo uses the diversity penalty selected through validation: **1.0**.

In [ ]:
from transformers.generation.utils import GenerationMixin
import transformers

# Diverse beam search is built into transformers up to 4.56 (the Hub route used from 4.57 fails on Windows).
if not hasattr(GenerationMixin, "_group_beam_search"):
    raise RuntimeError(
        f"transformers {transformers.__version__} no longer includes diverse beam search. "
        "Use the .venv-scoring kernel with transformers>=4.46,<4.57 (see 'Fix the transformers version' in SETUP_SCORING.md)."
    )


@torch.inference_mode()
def generate_dbs_candidates(abstract):
    inputs = tokenize_abstract(abstract)
    output_ids = bart_model.generate(
        **inputs,
        num_beams=NUM_BEAMS,
        num_beam_groups=NUM_BEAM_GROUPS,
        num_return_sequences=NUM_RETURN_SEQUENCES,
        diversity_penalty=DIVERSITY_PENALTY,
        do_sample=False,
        max_length=MAX_TARGET_LENGTH
    )
    return [
        tokenizer.decode(ids, skip_special_tokens=True).strip()
        for ids in output_ids
    ]


def show_candidates(_):
    abstract = abstract_input.value.strip()
    if not abstract:
        print("Enter an abstract first.")
        return
    candidates = generate_dbs_candidates(abstract)
    for i, candidate in enumerate(candidates, 1):
        display(Markdown(f"**Candidate {i}**\n\n{candidate}"))
    globals()["current_candidates"] = candidates


dbs_button = widgets.Button(description="Generate 5 DBS Candidates", button_style="primary")
dbs_button.on_click(show_candidates)
display(dbs_button)

## 7. Load AlignScore

AlignScore scores each candidate (claim) against the original abstract (context). It is a semantic alignment score, not a probability or a guarantee of factual correctness.

The following uses the commonly used AlignScore Python interface. If your experiment used a different version or wrapper, replace this cell with the exact loading code from your working evaluation script.

In [ ]:
# If not installed, follow the official AlignScore repository instructions:
# %pip install git+https://github.com/yuh-zha/AlignScore

import transformers
try:
    transformers.AdamW
except Exception:
    transformers.AdamW = torch.optim.AdamW   # AlignScore imports it; recent transformers removed it (only used for training)

from alignscore import AlignScore

if not os.path.isfile(ALIGNSCORE_CHECKPOINT):
    raise FileNotFoundError(
        f"AlignScore checkpoint not found: {ALIGNSCORE_CHECKPOINT}. "
        "Download it using the official AlignScore instructions and update the path."
    )

alignscore_scorer = AlignScore(
    model=ALIGNSCORE_MODEL,
    batch_size=8,
    device=DEVICE,
    ckpt_path=ALIGNSCORE_CHECKPOINT,
    evaluation_mode=ALIGNSCORE_EVALUATION_MODE
)
print("AlignScore loaded.")

## 8. Rerank candidates and select one

For each abstract, AlignScore scores all candidates against the same original abstract. The candidate with the highest AlignScore is selected.

In [ ]:
def rerank_candidates(abstract, candidates):
    contexts = [abstract] * len(candidates)
    claims = list(candidates)
    scores = alignscore_scorer.score(contexts=contexts, claims=claims)

    results = pd.DataFrame({
        "Candidate": [f"Candidate {i}" for i in range(1, len(candidates) + 1)],
        "Summary": candidates,
        "AlignScore": [float(s) for s in scores]
    }).sort_values("AlignScore", ascending=False).reset_index(drop=True)

    return results, results.iloc[0]


def run_pipeline(_):
    abstract = abstract_input.value.strip()
    if not abstract:
        print("Enter an abstract first.")
        return

    print("Generating greedy baseline...")
    baseline = generate_greedy(abstract)

    print("Generating DBS candidates...")
    candidates = generate_dbs_candidates(abstract)

    print("Scoring candidates with AlignScore...")
    scores, selected = rerank_candidates(abstract, candidates)

    display(Markdown("## Baseline: Greedy BART"))
    display(Markdown(baseline))

    display(Markdown("## AlignScore ranking"))
    display(scores[["Candidate", "AlignScore"]].round(4))

    display(Markdown("## Candidate summaries"))
    for _, row in scores.iterrows():
        display(Markdown(
            f"**{row['Candidate']} — AlignScore: {row['AlignScore']:.4f}**\n\n"
            f"{row['Summary']}"
        ))

    display(Markdown("## Selected summary: DBS + AlignScore"))
    display(Markdown(f"**{selected['Summary']}**"))

    globals()["demo_baseline"] = baseline
    globals()["demo_candidates"] = candidates
    globals()["demo_candidate_scores"] = scores
    globals()["demo_selected"] = selected["Summary"]


run_button = widgets.Button(
    description="Run Complete Summarization Pipeline",
    button_style="success"
)
run_button.on_click(run_pipeline)
display(run_button)

## 9. Aggregate test-set results

These are the project's reported aggregate test results, not scores calculated from the live abstract entered above. The selected DBS penalty is 1.0.

In [ ]:
test_results = pd.DataFrame({
    "Metric": ["SummaC", "ROUGE-L"],
    "Greedy BART": [0.7612, 0.3675],
    "DBS + AlignScore": [0.8753, 0.3658]
})
test_results["Difference"] = (
    test_results["DBS + AlignScore"] - test_results["Greedy BART"]
)

display(test_results.round(4))
print("Evaluated test abstracts: 615")

In [ ]:
# Plot each metric separately.
for metric, greedy, proposed in [
    ("SummaC", 0.7612, 0.8753),
    ("ROUGE-L", 0.3675, 0.3658)
]:
    plt.figure(figsize=(6, 4))
    plt.bar(["Greedy BART", "DBS + AlignScore"], [greedy, proposed])
    plt.title(f"Test-set {metric} comparison")
    plt.ylabel("Mean score")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

## 10. Discussion and presentation notes

- **SummaC:** Increased from 0.7612 to 0.8753 (+0.1141), indicating higher estimated factual consistency according to this metric.
- **ROUGE-L:** Decreased slightly from 0.3675 to 0.3658 (-0.0017), indicating very similar but marginally lower overlap with human-written TLDRs.
- The results suggest improved source consistency without an improvement in reference overlap.
- Automatic metrics do not guarantee that every selected summary is factually correct.

### Explain the roles clearly
- **DBS:** Generates diverse candidate summaries.
- **AlignScore:** Selects one candidate for each abstract.
- **Validation SummaC:** Helps select the diversity penalty configuration by comparing mean scores across validation examples.
- **Test SummaC and ROUGE-L:** Evaluate the frozen final method against greedy BART.

### Before presenting
- Confirm that the test evaluation used penalty 1.0, selected on validation.
- Explain why 615 abstracts were evaluated although the original test split contained 618.
- Ensure the BART and AlignScore checkpoints and preprocessing match your reported experiment.
- Do not present live AlignScore values as SummaC or ROUGE-L results.

## Troubleshooting

- **Hugging Face error:** Check the model repository name, permissions, and optional revision.
- **AlignScore error:** Check the package version, checkpoint path, and scoring settings against your original evaluation code.
- **CUDA out-of-memory:** Reduce AlignScore batch size, close other GPU workloads, or run models sequentially.
- **Grouped beam search error:** Verify your Transformers version supports grouped beam search and `diversity_penalty`.
- **Slow live inference:** Prepare a precomputed abstract and its outputs as a backup.

**Reproducibility:** Use the exact BART checkpoint, tokenizer, AlignScore checkpoint, and preprocessing settings from your experiment.